# EDA — Perfiles financieros sintéticos

**Objetivo:** describir qué variables contiene la base, cómo se distribuyen y qué controles básicos de calidad conviene observar antes de modelar.

> **Alcance:** los 30.000 registros son sintéticos y fueron generados con reglas definidas en `scripts/generar_perfiles_sinteticos.py`. No son respuestas de personas ni representan a la población colombiana. Las distribuciones y asociaciones que aparecen aquí describen únicamente el mecanismo de simulación.

## 1. Carga de datos y diccionario

El CSV se conserva en `data/raw/`. La descripción de su origen y los campos está en `docs/datos_sinteticos.md`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_FILE = Path("data/raw/perfiles_crediticios_sinteticos.csv")
DATA_URL = "https://raw.githubusercontent.com/catalinagaviriamet-spec/Modelo-de-Inclusi-n-Crediticia-Estudiantes-UdeM/main/data/raw/perfiles_crediticios_sinteticos.csv"
if DATA_FILE.exists():
    df = pd.read_csv(DATA_FILE)
    fuente = str(DATA_FILE.resolve())
else:
    # En Google Colab el repositorio no está descargado localmente: se lee el CSV publicado.
    df = pd.read_csv(DATA_URL)
    fuente = DATA_URL
print(f"Fuente: {fuente}")
print(f"Registros: {len(df):,} | Variables: {df.shape[1]}")
display(df.head())

## 2. Estructura y tipos de variables

Agrupamos las columnas por el tipo de información de la ficha. Esta vista ayuda a explicar qué puede preguntar el formulario y qué información representa cada bloque.

In [ ]:
grupos = {
    "Identificación sintética": ["id_persona"],
    "Edad": ["edad", "rango_edad"],
    "Hogar": ["tipo_hogar", "personas_hogar", "personas_a_cargo"],
    "Estudio y trabajo": ["nivel_estudio", "situacion_laboral"],
    "Ingresos y activos": ["tipo_ingreso", "ingreso_mensual_cop", "tiene_ingresos", "tiene_vivienda_propia", "tiene_vehiculo"],
    "Gastos y conectividad": ["pago_servicios_publicos_cop", "tipo_plan_celular"],
    "Experiencia y obligaciones": ["experiencia_crediticia", "meses_historial_crediticio", "deuda_mensual_cop", "ahorro_disponible_cop", "moras_ultimos_12_meses", "monto_solicitado_cop"],
}
resumen_grupos = pd.DataFrame([{"grupo": grupo, "cantidad_campos": len(cols), "campos": ", ".join(cols)} for grupo, cols in grupos.items()])
display(resumen_grupos)

perfil_columnas = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "nulos": df.isna().sum(),
    "valores_distintos": df.nunique(dropna=True),
})
display(perfil_columnas)

## 3. Revisión descriptiva de calidad

Revisamos identificadores duplicados, valores faltantes, rangos de edad y coherencia entre `edad` y `rango_edad`. Esto describe la consistencia del archivo; no valida que las reglas sintéticas representen comportamientos reales.

In [ ]:
def clasificar_edad(edad):
    if 18 <= edad <= 25:
        return "18-25"
    if 26 <= edad <= 40:
        return "26-40"
    if 41 <= edad <= 60:
        return "41-60"
    if 61 <= edad <= 74:
        return "61-74"
    return "fuera_de_rango"

control_calidad = pd.Series({
    "filas": len(df),
    "columnas": df.shape[1],
    "ids_duplicados": df["id_persona"].duplicated().sum(),
    "celdas_faltantes": int(df.isna().sum().sum()),
    "edad_minima": df["edad"].min(),
    "edad_maxima": df["edad"].max(),
    "edades_fuera_de_18_74": ((df["edad"] < 18) | (df["edad"] > 74)).sum(),
    "rango_edad_inconsistente": (df["rango_edad"] != df["edad"].map(clasificar_edad)).sum(),
    "sin_historial_con_meses_distintos_de_cero": ((df["experiencia_crediticia"] == "sin_experiencia") & (df["meses_historial_crediticio"] != 0)).sum(),
})
display(control_calidad.to_frame("resultado"))
display(df["rango_edad"].value_counts().rename_axis("rango_edad").to_frame("registros"))

## 4. Edad, hogar y educación

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df["edad"].plot.hist(bins=20, ax=axes[0], color="#4C78A8", edgecolor="white")
axes[0].set(title="Distribución de edad", xlabel="Edad", ylabel="Personas")
df["rango_edad"].value_counts().reindex(["18-25", "26-40", "41-60", "61-74"]).plot.bar(ax=axes[1], color="#72B7B2")
axes[1].set(title="Perfiles por rango de edad", xlabel="Rango", ylabel="Personas")
df["tipo_hogar"].value_counts().sort_values().plot.barh(ax=axes[2], color="#F2CF5B")
axes[2].set(title="Tipo de hogar", xlabel="Personas", ylabel="")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["nivel_estudio"].value_counts().sort_values().plot.barh(ax=axes[0], color="#B279A2")
axes[0].set(title="Nivel de estudio", xlabel="Personas", ylabel="")
df["personas_a_cargo"].value_counts().sort_index().plot.bar(ax=axes[1], color="#FF9DA6")
axes[1].set(title="Personas a cargo", xlabel="Número", ylabel="Registros")
plt.tight_layout()
plt.show()

## 5. Situación laboral e ingresos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["situacion_laboral"].value_counts().sort_values().plot.barh(ax=axes[0], color="#9D755D")
axes[0].set(title="Situación laboral", xlabel="Personas", ylabel="")
df["tipo_ingreso"].value_counts().sort_values().plot.barh(ax=axes[1], color="#4C78A8")
axes[1].set(title="Fuente de ingreso declarada", xlabel="Personas", ylabel="")
plt.tight_layout()
plt.show()

con_ingreso = df.loc[df["ingreso_mensual_cop"] > 0, "ingreso_mensual_cop"]
print(f"Personas con ingreso: {(df['tiene_ingresos'] == 1).sum():,} de {len(df):,} ({(df['tiene_ingresos'].mean() * 100):.1f}%)")
display(df[["ingreso_mensual_cop", "pago_servicios_publicos_cop", "deuda_mensual_cop", "ahorro_disponible_cop", "monto_solicitado_cop"]].describe().T.round(0))
fig, ax = plt.subplots(figsize=(8, 4))
con_ingreso.plot.hist(bins=35, ax=ax, color="#59A14F", edgecolor="white")
ax.set(title="Distribución de ingresos positivos (COP)", xlabel="Ingreso mensual, COP", ylabel="Registros")
ax.ticklabel_format(style="plain", axis="x")
plt.tight_layout()
plt.show()

## 6. Vivienda, vehículo, servicios y celular

In [ ]:
binarias = {
    "Vivienda propia": "tiene_vivienda_propia",
    "Tiene vehículo": "tiene_vehiculo",
    "Tiene ingresos": "tiene_ingresos",
}
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (titulo, columna) in zip(axes, binarias.items()):
    df[columna].map({0: "No", 1: "Sí"}).value_counts().reindex(["No", "Sí"]).plot.bar(ax=ax, color=["#E15759", "#59A14F"])
    ax.set(title=titulo, xlabel="", ylabel="Personas")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["tipo_plan_celular"].value_counts().reindex(["sin_linea", "prepago", "pospago"]).plot.bar(ax=axes[0], color="#F28E2B")
axes[0].set(title="Tipo de plan celular", xlabel="Tipo", ylabel="Registros")
axes[0].tick_params(axis="x", rotation=0)
df["pago_servicios_publicos_cop"].plot.hist(bins=30, ax=axes[1], color="#76B7B2", edgecolor="white")
axes[1].set(title="Pago mensual de servicios", xlabel="COP", ylabel="Registros")
axes[1].ticklabel_format(style="plain", axis="x")
plt.tight_layout()
plt.show()

## 7. Experiencia crediticia y solicitudes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df["experiencia_crediticia"].value_counts().sort_values().plot.barh(ax=axes[0], color="#4E79A7")
axes[0].set(title="Experiencia crediticia", xlabel="Personas", ylabel="")
df["meses_historial_crediticio"].plot.hist(bins=25, ax=axes[1], color="#EDC948", edgecolor="white")
axes[1].set(title="Meses de historial", xlabel="Meses", ylabel="Registros")
df["moras_ultimos_12_meses"].value_counts().sort_index().plot.bar(ax=axes[2], color="#E15759")
axes[2].set(title="Moras sintéticas recientes", xlabel="Número", ylabel="Registros")
plt.tight_layout()
plt.show()

resumen_credito = df.groupby("experiencia_crediticia").agg(
    personas=("id_persona", "count"),
    ingreso_mediano_cop=("ingreso_mensual_cop", "median"),
    deuda_mediana_cop=("deuda_mensual_cop", "median"),
    solicitud_mediana_cop=("monto_solicitado_cop", "median"),
).round(0)
display(resumen_credito)

## 8. Relaciones entre variables económicas

Estos gráficos ilustran las asociaciones que quedaron incorporadas en la simulación. No deben interpretarse como correlaciones empíricas para tomar decisiones sobre clientes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(df["ingreso_mensual_cop"], df["deuda_mensual_cop"], alpha=0.18, s=12, color="#4C78A8")
axes[0].set(title="Ingreso y deuda mensual", xlabel="Ingreso mensual (COP)", ylabel="Deuda mensual (COP)")
axes[0].ticklabel_format(style="plain")
axes[1].scatter(df["ingreso_mensual_cop"], df["monto_solicitado_cop"], alpha=0.18, s=12, color="#F28E2B")
axes[1].set(title="Ingreso y monto solicitado", xlabel="Ingreso mensual (COP)", ylabel="Monto solicitado (COP)")
axes[1].ticklabel_format(style="plain")
plt.tight_layout()
plt.show()

cols_numericas = ["edad", "personas_hogar", "personas_a_cargo", "ingreso_mensual_cop", "pago_servicios_publicos_cop", "deuda_mensual_cop", "ahorro_disponible_cop", "monto_solicitado_cop"]
display(df[cols_numericas].corr().round(2))

## 9. Conclusiones del EDA

- La tabla contiene perfiles ficticios a nivel de persona y organiza variables de edad, hogar, educación, trabajo, ingresos, acceso a activos y servicios, y experiencia crediticia.
- `rango_edad` resume la edad en cuatro grupos; el historial crediticio puede ser cero cuando la persona indica no tener experiencia.
- Los campos monetarios están expresados en COP y los indicadores binarios usan 0/1.
- Los gráficos reflejan las reglas y rangos elegidos para generar datos sintéticos. No describen prevalencias reales ni prueban que una variable cause mayor o menor capacidad de pago.
- La base no contiene puntajes derivados ni una ruta recomendada; esos elementos se definirán en una etapa posterior del proyecto.

## 10. Alcance y próximos pasos

Este EDA documenta el contenido y la construcción de la versión sintética actual. Antes de construir reglas o modelos de decisión, el equipo debe definir criterios de capacidad de pago, cómo medir educación financiera, qué significa “cliente maduro” y cómo validar las rutas. Para cualquier uso con personas reales se requerirían datos obtenidos con autorización y evaluación de privacidad, calidad y equidad.